In [78]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd

In [79]:
prison = Prison()
actions = prison.Get_Actions()

strategies = {
    0: Random_Strategy(0, actions),
    1: Always_Cooperate(1, actions),
    2: Always_Betray(2, actions),
    3: Random_Strategy(3, actions, p_coop=0.1),
    4 : Patient_Unforgiving(4, actions),
}

In [80]:
gm = Game_Master(prison, strategies=strategies, duel_size=2)
duel_matrix, rewards = gm.Tournament(4, Game_Master.Game_Type.All_Vs_All, True)
rewards.Sort_Total_Rewards()

In [81]:
total_rewards = rewards.Get_Total_Rewards()
rewards_per_name = {str(strategies[i]):total_rewards[i] for i in total_rewards.keys()}
# rewards_per_name = pd.DataFrame(rewards_per_name, index=[0, -1])

duel_rewards = rewards.Get_All_Duel_Rewards()
winner = next(iter(total_rewards))

In [82]:
print(rewards_per_name)

{'Always_Betray': 52, 'Random_Strategy (p_coop=0.1)': 45, 'Patient_Unforgiving (patience=1)': 29, 'Always_Cooperate': 27, 'Random_Strategy (p_coop=0.5)': 20}


In [83]:
df = pd.DataFrame.from_dict(rewards_per_name, orient="index", columns=["Total Reward"])
df.index.name = "Name"
df

,Total Reward
Name,
Always_Betray,52
Random_Strategy (p_coop=0.1),45
Patient_Unforgiving (patience=1),29
Always_Cooperate,27
Random_Strategy (p_coop=0.5),20


In [84]:
strat_names = [str(s) for s in strategies.values()]

df = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    df.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2], scores[s1])
    df.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s1], scores[s2])

for s in strat_names:
    df.loc[s, s] = (0, 0)

In [85]:
binary_df = df.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for s in strat_names:
    binary_df.loc[s, s] = float("NaN")

In [86]:
def color_cell(x):
    if not isinstance(x, tuple):
        return ""
    if x[0] > x[1]:
        return "background-color: green"
    elif x[0] < x[1]:
        return "background-color: darkred"
    else:
        return "background-color: gray"

styled = df.style.map(color_cell)
styled.to_html("matrix.html")
display(styled)

,Random_Strategy (p_coop=0.5),Always_Cooperate,Always_Betray,Random_Strategy (p_coop=0.1),Patient_Unforgiving (patience=1)
Random_Strategy (p_coop=0.5),"(0, 0)","(12, 12)","(1, 16)","(1, 16)","(6, 11)"
Always_Cooperate,"(12, 12)","(0, 0)","(0, 20)","(3, 18)","(12, 12)"
Always_Betray,"(16, 1)","(20, 0)","(0, 0)","(8, 3)","(8, 3)"
Random_Strategy (p_coop=0.1),"(16, 1)","(18, 3)","(3, 8)","(0, 0)","(8, 3)"
Patient_Unforgiving (patience=1),"(11, 6)","(12, 12)","(3, 8)","(3, 8)","(0, 0)"
